# Step 1: Create Azure Identities

![Azure](https://img.shields.io/badge/Azure-0078D4?logo=microsoftazure&logoColor=white)
![Azure CLI](https://img.shields.io/badge/Azure_CLI-0078D4?logo=microsoftazure&logoColor=white)

This notebook walks you through creating Azure identities (Service Principal or Managed Identity) for GitHub Actions OIDC authentication.

**What this notebook does:**
1. Validates prerequisites (Azure CLI installed and authenticated)
2. Collects your configuration (GitHub owner, environments, resource groups)
3. Creates Service Principals **or** Managed Identities (you choose)
4. Assigns RBAC roles scoped to resource groups
5. Saves configuration to `config.json` for use by subsequent notebooks

> **Works with both** GitHub organizations and personal accounts.

## Identity Type Comparison

| | Service Principal | User-Assigned Managed Identity |
|---|---|---|
| **Azure AD admin required** | Yes (`Application.ReadWrite.All`) | No — just `Contributor` on the resource group |
| **Client secrets can exist** | Yes (even if unused with OIDC) | No — secrets physically cannot be created |
| **IaC support** | Requires AzureAD Terraform provider | First-class Azure resource (Terraform/Bicep/ARM) |
| **Cleanup** | Can become orphaned in Azure AD | Deleting the resource cleans up everything |
| **Script flag** | (default) | `--managed-identity` |

## Prerequisites Check

Run the cell below to verify Azure CLI is installed and authenticated.

In [ ]:
import subprocess, json, sys, os

def run_cmd(cmd, capture=True, check=True):
    """Run a shell command and return the result."""
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
        return None
    return result

# Check Azure CLI
r = run_cmd("az version --output json", check=False)
if r and r.returncode == 0:
    ver = json.loads(r.stdout)
    print(f"✓ Azure CLI installed: {ver.get('azure-cli', 'unknown')}")
else:
    print("✗ Azure CLI not found. Install: https://aka.ms/installazurecli")
    raise SystemExit("Azure CLI is required. Install it and re-run this cell.")

# Check Azure authentication
r = run_cmd("az account show --output json", check=False)
if r and r.returncode == 0:
    acct = json.loads(r.stdout)
    print(f"✓ Authenticated as: {acct.get('user', {}).get('name', 'unknown')}")
    print(f"  Subscription: {acct.get('name', 'unknown')} ({acct.get('id', 'unknown')})")
else:
    print("✗ Not authenticated. Run 'az login' in a terminal first.")
    raise SystemExit("Azure CLI authentication required. Run 'az login' and re-run this cell.")

print("\n✓ All prerequisites met.")

## Configuration

Enter your GitHub owner (organization name or personal username), subscription ID, and environment details.

In [ ]:
# --- Configuration ---
# Modify these values for your setup

GITHUB_OWNER = input("GitHub owner (org name or personal username): ").strip()
SUBSCRIPTION_ID = input("Azure subscription ID: ").strip()
env_input = input("Environments (comma-separated, default: dev, staging, production): ").strip()
ENVIRONMENTS = [e.strip() for e in env_input.split(",") if e.strip()] if env_input else ["dev", "staging", "production"]

print(f"\nGitHub owner: {GITHUB_OWNER}")
print(f"Subscription: {SUBSCRIPTION_ID}")
print(f"Environments: {ENVIRONMENTS}")

In [ ]:
# --- Resource Groups ---
# Enter the resource group name for each environment.
# These are the RGs your GitHub Actions workflows will deploy to.

RESOURCE_GROUPS = {}
for env in ENVIRONMENTS:
    rg = input(f"Resource group for '{env}' environment: ").strip()
    RESOURCE_GROUPS[env] = rg

print("\nResource groups:")
for env, rg in RESOURCE_GROUPS.items():
    print(f"  {env}: {rg}")

In [ ]:
# --- Identity Type ---
# Choose: 'sp' for Service Principal, 'mi' for Managed Identity

IDENTITY_TYPE = input("Identity type ('sp' for Service Principal, 'mi' for Managed Identity): ").strip().lower()
while IDENTITY_TYPE not in ('sp', 'mi'):
    IDENTITY_TYPE = input("Please enter 'sp' or 'mi': ").strip().lower()

MI_RESOURCE_GROUP = ""
if IDENTITY_TYPE == "mi":
    MI_RESOURCE_GROUP = input("Resource group for managed identities (e.g., 'infra-rg'): ").strip()

print(f"\nIdentity type: {'Service Principal' if IDENTITY_TYPE == 'sp' else 'Managed Identity'}")
if MI_RESOURCE_GROUP:
    print(f"MI resource group: {MI_RESOURCE_GROUP}")

## Create Identities

Run the appropriate cell below based on your chosen identity type.

> **Already have identities?** If a Service Principal or Managed Identity with the expected name already exists, the cell will detect it and reuse it instead of failing — no duplicates are created.

### Option A: Service Principal

Creates one SP per environment, scoped to the environment's resource group.

In [ ]:
# --- Option A: Create Service Principals ---
# Skip this cell if you chose Managed Identity

if IDENTITY_TYPE != "sp":
    print("Skipped — you selected Managed Identity. Run the next cell instead.")
else:
    sp_results = {}
    for env in ENVIRONMENTS:
        name = f"github-actions-{env}"
        rg = RESOURCE_GROUPS[env]
        scope = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{rg}"

        # Check if SP already exists
        print(f"\nChecking for existing SP '{name}'...")
        r_existing = run_cmd(f'az ad sp list --display-name "{name}" --query "[0]" --output json', check=False)
        if r_existing and r_existing.returncode == 0 and r_existing.stdout.strip() not in ("", "null"):
            data = json.loads(r_existing.stdout)
            client_id = data.get("appId", "")
            r2 = run_cmd(f'az ad sp show --id "{client_id}" --query id -o tsv', check=False)
            object_id = r2.stdout.strip() if r2 and r2.returncode == 0 else "UNKNOWN"
            # Get tenant ID
            r_tenant = run_cmd("az account show --query tenantId -o tsv", check=False)
            tenant = r_tenant.stdout.strip() if r_tenant and r_tenant.returncode == 0 else ""
            sp_results[env] = {
                "name": name,
                "client_id": client_id,
                "object_id": object_id,
                "tenant": tenant
            }
            print(f"  ⊘ SP '{name}' already exists — reusing it.")
            print(f"    clientId={client_id}, objectId={object_id}")
            continue

        print(f"Creating SP '{name}' scoped to {rg}...")
        cmd = (
            f'az ad sp create-for-rbac --name "{name}" '
            f'--role contributor --scopes "{scope}" --output json'
        )
        r = run_cmd(cmd, check=False)
        if r and r.returncode == 0:
            data = json.loads(r.stdout)
            client_id = data["appId"]
            # Get the Object ID (needed for federated credentials)
            r2 = run_cmd(f'az ad sp show --id "{client_id}" --query id -o tsv', check=False)
            object_id = r2.stdout.strip() if r2 and r2.returncode == 0 else "UNKNOWN"
            sp_results[env] = {
                "name": name,
                "client_id": client_id,
                "object_id": object_id,
                "tenant": data.get("tenant", "")
            }
            print(f"  ✓ Created: clientId={client_id}, objectId={object_id}")
        else:
            print(f"  ✗ Failed: {r.stderr.strip() if r else 'unknown error'}")

    if sp_results:
        print("\n--- Summary ---")
        for env, info in sp_results.items():
            print(f"  {env}: clientId={info['client_id']}, objectId={info['object_id']}")

### Option B: Managed Identity

Creates one User-Assigned Managed Identity per environment, then assigns RBAC roles.

In [ ]:
# --- Option B: Create Managed Identities ---
# Skip this cell if you chose Service Principal

if IDENTITY_TYPE != "mi":
    print("Skipped — you selected Service Principal. Run the previous cell instead.")
else:
    import time

    MI_LOCATION = input(f"Azure region for MI resource group (default: eastus): ").strip() or "eastus"

    # Ensure the MI resource group exists
    print(f"Ensuring resource group '{MI_RESOURCE_GROUP}' exists...")
    r = run_cmd(
        f'az group create --name "{MI_RESOURCE_GROUP}" --location "{MI_LOCATION}" --output json',
        check=False
    )
    if r and r.returncode == 0:
        print(f"  ✓ Resource group ready: {MI_RESOURCE_GROUP}")
    else:
        print(f"  ✗ Failed to create/verify resource group: {r.stderr.strip() if r else ''}")

    mi_results = {}
    need_rbac = []  # track which envs need new RBAC assignments

    for env in ENVIRONMENTS:
        name = f"github-actions-{env}"

        # Check if identity already exists
        print(f"\nChecking for existing identity '{name}'...")
        r_existing = run_cmd(
            f'az identity show --name "{name}" --resource-group "{MI_RESOURCE_GROUP}" --output json',
            check=False
        )
        if r_existing and r_existing.returncode == 0 and r_existing.stdout.strip():
            data = json.loads(r_existing.stdout)
            client_id = data["clientId"]
            principal_id = data["principalId"]
            mi_results[env] = {
                "name": name,
                "client_id": client_id,
                "principal_id": principal_id
            }
            print(f"  ⊘ Identity '{name}' already exists — reusing it.")
            print(f"    clientId={client_id}, principalId={principal_id}")
            continue

        print(f"Creating managed identity '{name}'...")
        cmd = (
            f'az identity create --name "{name}" '
            f'--resource-group "{MI_RESOURCE_GROUP}" --output json'
        )
        r = run_cmd(cmd, check=False)
        if r and r.returncode == 0:
            data = json.loads(r.stdout)
            client_id = data["clientId"]
            principal_id = data["principalId"]
            mi_results[env] = {
                "name": name,
                "client_id": client_id,
                "principal_id": principal_id
            }
            need_rbac.append(env)
            print(f"  ✓ Created: clientId={client_id}, principalId={principal_id}")
        else:
            print(f"  ✗ Failed: {r.stderr.strip() if r else 'unknown error'}")

    # Assign RBAC roles only for newly created identities
    if need_rbac:
        print("\n--- Waiting 30s for Azure AD replication to propagate... ---")
        time.sleep(30)

        print("--- Assigning RBAC roles ---")
        for env in need_rbac:
            info = mi_results[env]
            rg = RESOURCE_GROUPS[env]
            scope = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{rg}"
            assigned = False
            for attempt in range(1, 4):
                cmd = (
                    f'az role assignment create --assignee-object-id "{info["principal_id"]}" '
                    f'--assignee-principal-type ServicePrincipal '
                    f'--role contributor --scope "{scope}" --output json'
                )
                r = run_cmd(cmd, check=False)
                if r and r.returncode == 0:
                    print(f"  ✓ {env}: Contributor on {rg}")
                    assigned = True
                    break
                else:
                    print(f"  ⟳ {env}: Attempt {attempt}/3 failed, retrying in 15s...")
                    time.sleep(15)
            if not assigned:
                print(f"  ✗ {env}: Failed after 3 attempts — {r.stderr.strip() if r else ''}")
    elif mi_results:
        print("\n--- All identities already existed, skipping RBAC assignment. ---")

    if mi_results:
        print("\n--- Summary ---")
        for env, info in mi_results.items():
            print(f"  {env}: clientId={info['client_id']}, principalId={info['principal_id']}")

## Save Configuration

Save the collected configuration to `config.json` for use by subsequent notebooks.

In [ ]:
# --- Save config.json ---
from pathlib import Path

config = {
    "org_name": GITHUB_OWNER,
    "subscription_id": SUBSCRIPTION_ID,
    "environments": ENVIRONMENTS,
    "resource_groups": RESOURCE_GROUPS,
    "identity_type": IDENTITY_TYPE,
}

if IDENTITY_TYPE == "sp" and 'sp_results' in dir() and sp_results:
    config["identities"] = sp_results
    # Get tenant ID from first SP
    first = next(iter(sp_results.values()))
    config["tenant_id"] = first.get("tenant", "")
elif IDENTITY_TYPE == "mi" and 'mi_results' in dir() and mi_results:
    config["identities"] = mi_results
    config["mi_resource_group"] = MI_RESOURCE_GROUP
    # Get tenant ID from Azure account
    r = run_cmd("az account show --query tenantId -o tsv", check=False)
    config["tenant_id"] = r.stdout.strip() if r and r.returncode == 0 else ""

config_path = Path.cwd() / "config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✓ Configuration saved to {config_path}")
print(json.dumps(config, indent=2))

## Next Steps

Proceed to **[02_setup_credentials.ipynb](02_setup_credentials.ipynb)** to create federated credentials linking these identities to your GitHub repositories.